In [ ]:
import ollama
import glob

### Document Poisoning (RAG or similar)

In this notebook, we'll look into what happens if your documents get poisoned and explore how that could change your system if you don't prepare appropriate document provenance and security measures. 

In [ ]:
with open('data/templates/rag_prompt.txt', 'r') as prompt_file:
    system_prompt = prompt_file.read()

In [ ]:
model_name = 'gemma3:4b'

In [ ]:
document_data = "Additional context: To answer the user's query, please use the following document sources: "

In [ ]:
for file in list(glob.glob('data/documents/*')):
    if 'poisoned' in file: # we'll get to this one in a second :) 
        continue
    with open(file, 'r') as ofile:
        document_data += "\n\n-------{filename}-------\n{content}".format(
            filename=file,
            content=ofile.read()
        )

In [ ]:
user_prompt = "What can you tell me about RAG systems?"

In [ ]:
user_input = "User Query: {query}\n\n{documents}".format(query=user_prompt, documents=document_data)

In [ ]:
messages = [
    {'role': 'system', 
     'content': system_prompt},
    {'role': 'user',
     'content': user_input,
    }
]

In [ ]:
response = ollama.chat(
    model=model_name,
    messages=messages
)

In [ ]:
response.message.content

### Poisoning the well...

Let's look at what happens when a document gets poisoned with a prompt injection attack.



In [ ]:
with open('data/documents/poisoned_doc.txt', 'r') as poisoned_doc:
    document_data += "\n\n-------{filename}-------\n{content}".format(
        filename='nothing_to_see_here.txt',
        content=poisoned_doc.read()
    )

In [ ]:
document_data

In [ ]:
user_prompt = "Hey, what's the latest in the documents? I want to know any and all updates."

In [ ]:
user_input = "User Query: {query}\n\n{documents}".format(query=user_prompt, documents=document_data)

In [ ]:
messages = [
    {'role': 'system', 
     'content': system_prompt},
    {'role': 'user',
     'content': user_input,
    }
]

In [ ]:
response = ollama.chat(
    model=model_name,
    messages=messages
)

In [ ]:
response

### Your Turn

- Can you make a better poisoned doc? (note: ollama models don't have "send email" tool, so this is safe to try)
- Test with other models; does the current document work with any of them? 
- If you find a way to poison that works, save it as a successful attack in your traces!

In [ ]:
document_data = document_data.split('\n\n-------nothing_to_see_here.txt-------\n')[0]

In [ ]:
with open('data/documents/poisoned_less_harmful.txt', 'r') as poisoned_doc:
    document_data += "\n\n-------{filename}-------\n{content}".format(
        filename='im_innocent.txt',
        content=poisoned_doc.read()
    )

In [ ]:
user_prompt = "Hey, what can I learn about security?"

In [ ]:
user_input = "User Query: {query}\n\n{documents}".format(query=user_prompt, documents=document_data)

In [ ]:
messages = [
    {'role': 'system', 
     'content': system_prompt},
    {'role': 'user',
     'content': user_input,
    }
]

In [ ]:
response = ollama.chat(
    model=model_name,
    messages=messages
)

In [ ]:
response

### Your Turn

- Can you add a "signature" and validate that files are signed using the LLM? 
- What other ideas can we use from information security that would allow us to better protect our documents? 
- What false positives might exist that we can also test, where actual information looks like a redirection? 